In [ ]:
import csv
import pandas as pd
import numpy as np
import scanpy as sc

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
# import dataframe_image as dfi

import scipy as sp
from scipy import stats as st
import statsmodels.stats.anova as anova 
from statsmodels.stats.multicomp import pairwise_tukeyhsd
#from statannot import add_stat_annotation

#import openpyxl

In [ ]:
sc.set_figure_params(dpi=100, color_map = 'viridis_r')
sc.settings.verbosity = 1
sc.logging.print_header()

In [ ]:
adata = sc.read("/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/integrated-aging-soupxed_withQpctl.h5ad", var_names="gene_symbols")

In [ ]:
sub4 = sc.read("/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub4_integrated-aging-soupxedQpctl.h5ad", var_names="gene_symbols")
sub12 = sc.read("/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub12_integrated-aging-soupxedQpctl.h5ad", var_names="gene_symbols")
sub14 = sc.read("/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub14_integrated-aging-soupxedQpctl.h5ad", var_names="gene_symbols")

In [ ]:
#set(sub12.obs['subclusters'])
sub4.obs

In [ ]:
sub12.obs

In [ ]:
sub14.obs

## Sample - Condition mapping

In [ ]:
value_mapping = {
    'SG18': 'ctrl',
    'SG20': 'age',
    'SG24': 'sAct',
    'SG26': 'DR',
    'SG28': 'combi',
    'sg18': 'ctrl',
    'sg20': 'age',
    'sg24': 'sAct',
    'sg26': 'DR',
    'sg28': 'combi',
    'rgzj1': 'ctrl',
    'rgzj2': 'age',
    'rgzj3': 'sAct',
    'rgzj4': 'DR',
    'rgzj5': 'combi'
}

adata.obs['sample'].replace(value_mapping, inplace=True)
#deg_sig['group'].replace(value_mapping, inplace=True)

Add all subclusters in adata 'subclutsers' column

In [ ]:
for sub in [sub4, sub12, sub14]:
    merged = sub.obs['clusters'].astype(str) + "." + sub.obs['subclusters'].astype(str)
    adata.obs.loc[sub.obs_names, 'mergedClusters'] = merged

Merge Clusters and Subclusters

In [ ]:
#adata.obs['mergedClusters']= adata.obs['subclusters']
adata.obs['mergedClusters']= adata.obs['mergedClusters'].combine_first(adata.obs['clusters'])
set(adata.obs['mergedClusters'])

# Plot UMAP

In [ ]:
#rc_context is used for the figure size, in this case 4x4
adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['clusters', 'mergedClusters'], wspace=0.4
               #, save = 'Umap_pig.png'
              )

In [ ]:
annotation = {
    "0": "PT-1", 
    "1": "TAL-1",
    "2": "DCT/CNT",
    "3": "PC_CNT",
    "4.0": "EC-2",
    "4.1": "EC-3",  
    "4.2": "EC-4",  
    "4.3": "EC-1(gEC)",  
    "4.4": "EC-5",  
    "4.5": "EC-6",  
    "5": "TAL-2",
    "6": "FIB",
    "7": "PODO",
    "8": "IC-B",
    "9": "Undef",    
    "10": "IC-A",
    "11": "PT-2",
    "12.0": "Uro",
    "12.1": "Undef",
    "12.2": "ATL",
    "12.3": "PC",
    "13": "vSMC/MC",
    "14.0": "IMM",
    "14.1": "IMM",
    "14.2": "IMM",
    "15": "PECs?",
    "16": "PROLIF",
    "17": "ADIPO"
}

In [ ]:
sub4.obs['subclusters'] = sub4.obs['clusters'].astype(str)+"."+sub4.obs['subclusters'].astype(str)
sub4.obs['subclusters']

In [ ]:
adata.obs["celltype"] = adata.obs["mergedClusters"].replace(annotation)
#sub4.obs["celltype"] = adata.obs["subclusters"].replace(annotation)
#sub4.obs["celltype"]

set(adata.obs["mergedClusters"])
set(adata.obs["celltype"])

# Cell Type UMAP

In [ ]:
#rc_context is used for the figure size, in this case 4x4
adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['celltype'], wspace=0.4)
    
 #   sc.pl.umap(sub4, color=['celltype'], wspace=0.4
 #              , save = '/data/projects/sonja/pig_stuff/paper_figs_pdf/Umap_pig.pdf')

# Celltype Composition

In [ ]:
adata_glom = adata[adata.obs['celltype'].isin(['PODO', 'vSMC/MC', 'PECs?', 'EC-1(gEC)'])]
adata_rest = adata[~adata.obs['celltype'].isin(['PODO', 'vSMC/MC', 'PECs?', 'EC-1(gEC)'])]

set(adata_rest.obs['celltype'])
#adata_rest.obs

In [ ]:
tmp = pd.crosstab(adata.obs['celltype'],adata.obs['sample'], normalize='columns').T.plot(kind='bar', stacked=True)
tmp.legend(title='celltype', bbox_to_anchor=(1.5, 1),loc='upper right')

In [ ]:
tmp = pd.crosstab(adata_glom.obs['celltype'],adata.obs['sample'], normalize='columns').T.plot(kind='bar', stacked=True)
tmp.legend(title='celltype', bbox_to_anchor=(1.5, 1),loc='upper right')

In [ ]:
tmp = pd.crosstab(adata_rest.obs['celltype'],adata.obs['sample'], normalize='columns').T.plot(kind='bar', stacked=True)
tmp.legend(title='celltype', bbox_to_anchor=(1.5, 1),loc='upper right')

# Write Output 

In [ ]:
outfile = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/annotated-aging-soupxedQpctl.h5ad'
adata.write(outfile)

In [ ]:
sc.pl.umap(adata, color=['mergedClusters', 'celltype'], 
           #legend_loc='on data',
           frameon=False, legend_fontsize=10, legend_fontoutline=1)

# Investigate expressions: Qpctl, Qpct, Ccl2

In [ ]:
genes_of_interest = ["Qpct", "Qpctl", "Ccl2"]
present_genes = [gene for gene in genes_of_interest if gene in adata.var_names]

# Dotplot across sample groups
#standard_scale='var': Each gene (variable) is scaled independently across all groups
sc.pl.dotplot(adata, var_names=present_genes, groupby="sample", swap_axes=True, standard_scale="var", 
              title="Gene Expression in Samples")

#standard_scale='group': Each group (cluster/category) is scaled independently across all genes
sc.pl.dotplot(adata, var_names=present_genes, groupby="sample", swap_axes=True, standard_scale="group", 
              title="Gene Expression in Samples")

# Dotplot across celltype groups
#standard_scale='var': Each gene (variable) is scaled independently across all groups
sc.pl.dotplot(adata, var_names=present_genes, groupby="celltype", swap_axes=True, standard_scale="var", 
              title="Gene Expression in Cell Types")

# standard_scale='group': Each group (cluster/category) is scaled independently across all genes.
sc.pl.dotplot(adata, var_names=present_genes, groupby="celltype", swap_axes=True, standard_scale="group", 
              title="Gene Expression in Cell Types")

In [ ]:
samples = adata.obs['sample'].unique()

for sample in samples:
    adata_sample = adata[adata.obs['sample'] == sample]
    # standard_scale='var': Each gene (variable) is scaled independently across all groups. For every gene, the minimum mean expression across groups is subtracted, and the result is divided by the maximum. This normalizes each gene's expression to the range $$$$ across the plotted groups.
    sc.pl.dotplot(adata_sample, var_names=present_genes, groupby='celltype', swap_axes=True, standard_scale='var',
                  title=f'Gene Expression in Cell Types - Sample {sample}')
    # standard_scale='group': Each group (cluster/category) is scaled independently across all genes. For every group, the minimum mean expression across all plotted genes is subtracted, and the result is divided by the maximum. This normalizes each group's gene expression profile to the range $$$$, making it easier to compare which genes are most expressed within each group.
    sc.pl.dotplot(adata_sample, var_names=present_genes, groupby='celltype', swap_axes=True, standard_scale='group',
                  title=f'Gene Expression in Cell Types - Sample {sample}')